# 10 — Stealing-Guided Synonym Substitution: V1 Multi-Ratio / Multi-Split

**Goal.** Compare four word-level synonym-substitution methods across **10 stratified train/test splits** and **five nominal replacement ratios** (`0.1, 0.2, 0.3, 0.4, 0.5`).

| Method | Word positions | WordNet synonym |
|---|---|---|
| `markllm` | MarkLLM (reproduced notebook 05) | MarkLLM |
| `guided` | Highest learned gain | Lowest WM-association score |
| `random_synonyms_guided_positions` | Same positions as `guided` | Random WordNet synonym |
| `fully_random` | Random eligible positions | Random WordNet synonym |

- **Per split:** learn on 400 matched WM/NoWM pairs; evaluate on 200 other WM responses. Keep the pairing and dataset proportions.
- **Detector:** the same local detector as notebook 03 (`src/detection.py`), evaluation only. No secret watermark metadata is used to choose replacements.
- **Budget:** the three newly implemented attacks have exactly the same **actual number of word replacements for each text**. MarkLLM uses the same *nominal* ratio, but its reported `n_replaced` is not guaranteed to count actual edits; we do **not** claim equal effective budgets with MarkLLM.
- **ASR:** among responses detected **before** attack, fraction not detected **after** attack.
- **Reproducibility:** split-specific results are saved to JSONL after each completed split. Rerunning skips completed splits after input/configuration checks.

**Scientific limitation:** the ten train/test splits overlap; their ASR standard deviation measures descriptive stability, not ten independent replications or a confirmatory p-value. The original fixed test set has already informed method development; these analyses are exploratory. Semantic preservation is not enforced.

## 1. Setup and configuration

In [1]:
import hashlib
import json
import math
import random
import re
import sys
import time
from collections import Counter, defaultdict
from functools import lru_cache
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from nltk.corpus import wordnet
from tqdm import tqdm


def find_project_root(start: Path) -> Path:
    for folder in [start, *start.parents]:
        if (folder / "src" / "detection.py").is_file() and (
            folder / "results" / "Cleaned Data"
        ).is_dir():
            return folder
    raise FileNotFoundError(
        "Cannot find the project root. Open the notebook from the project or notebooks/ folder."
    )


ROOT = find_project_root(Path.cwd().resolve())
sys.path.insert(0, str(ROOT / "src"))
from detection import detect, load_detector, load_jsonl, save_jsonl

SPLIT_SEEDS = [42, 100, 101, 102, 103, 104, 105, 106, 107, 108]
RATIOS = [0.1, 0.2, 0.3, 0.4, 0.5]
MIN_SUPPORT = 10         # Same as notebook 09; threshold experiments are separate.
ALPHA = 0.5              # Additive smoothing, same as notebook 09.
ATTACK_VERSION = "word_unigrams_v1_ratio_sweep_2026_11"
METHODS = [
    "markllm",
    "guided",
    "random_synonyms_guided_positions",
    "fully_random",
]

OUT_DIR = ROOT / "results" / "attacks" / "stealing_guided_v1_improvements" / "ratio_sweep_support10"
SPLITS_DIR = OUT_DIR / "splits"
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

# Word-level regex used in notebook 09 (not the LLaDA tokenizer).
WORD_PATTERN = re.compile(r"[a-z]+(?:'[a-z]+)?")

try:
    wordnet.synsets("important")
except LookupError as exc:
    raise RuntimeError(
        "NLTK WordNet is missing in this kernel; install its corpus before proceeding."
    ) from exc

print("ROOT:", ROOT)
print("OUTPUT:", OUT_DIR)
print("Splits:", len(SPLIT_SEEDS), "| Ratios:", RATIOS, "| Methods:", METHODS)

ROOT: /Users/paulmicheli/Desktop/Unine/MA1/Trustworthy/Trustworhty_AI_Watermarking
OUTPUT: /Users/paulmicheli/Desktop/Unine/MA1/Trustworthy/Trustworhty_AI_Watermarking/results/attacks/stealing_guided_v1_improvements/ratio_sweep_support10
Splits: 10 | Ratios: [0.1, 0.2, 0.3, 0.4, 0.5] | Methods: ['markllm', 'guided', 'random_synonyms_guided_positions', 'fully_random']


## 2. Load and validate the paired data and MarkLLM results

In [2]:
WM_PATH = ROOT / "results" / "Cleaned Data" / "watermarked_600.jsonl"
NOWM_PATH = ROOT / "results" / "Cleaned Data" / "unwatermarked_600.jsonl"
BASELINE_PATH = ROOT / "results" / "attacks" / "baseline_original.jsonl"
SYN_DIR = ROOT / "results" / "attacks" / "synonym_substitution"

wm = load_jsonl(WM_PATH)
no_wm = load_jsonl(NOWM_PATH)
baseline_rows = load_jsonl(BASELINE_PATH)
assert len(wm) == len(no_wm) == len(baseline_rows) == 600

baseline_by_idx = {int(row["idx"]): row for row in baseline_rows}
assert set(baseline_by_idx) == set(range(600)), "Baseline indices must be 0..599."


def dataset_id(row):
    return row["dataset_name"].split("_batch_")[0]


groups = defaultdict(list)
for idx in range(600):
    assert wm[idx]["prompt"] == no_wm[idx]["prompt"]
    assert wm[idx]["prompt"] == baseline_by_idx[idx]["prompt"]
    assert dataset_id(wm[idx]) == dataset_id(no_wm[idx])
    assert wm[idx]["completion"] and no_wm[idx]["completion"]
    groups[dataset_id(wm[idx])].append(idx)

assert len(groups) == 3 and sorted(map(len, groups.values())) == [200, 200, 200], (
    f"Expected three datasets with 200 prompts each, got: "
    f"{ {k: len(v) for k, v in groups.items()} }"
)

markllm_by_ratio = {}
markllm_paths = {}
for ratio in RATIOS:
    path = SYN_DIR / f"synonym_substitution_r{ratio:.1f}.jsonl"
    assert path.is_file(), f"Missing MarkLLM baseline: {path}"
    records = load_jsonl(path)
    lookup = {int(row["idx"]): row for row in records}
    assert len(records) == len(lookup) == 600
    assert set(lookup) == set(range(600))
    for idx in range(600):
        record = lookup[idx]
        assert record["original"] == wm[idx]["completion"], (ratio, idx)
        assert bool(record["original_detected"]) == bool(baseline_by_idx[idx]["detected"])
        assert abs(float(record["ratio"]) - ratio) < 1e-9
    markllm_by_ratio[ratio] = lookup
    markllm_paths[ratio] = path

print("Dataset sizes:", {k: len(v) for k, v in sorted(groups.items())})
print("Baseline originally detected:", sum(bool(r["detected"]) for r in baseline_rows), "/600")
print("MarkLLM records verified:", len(RATIOS), "ratios x 600 responses")

Dataset sizes: {'2-1_longform_qa': 200, '2-2_finance_qa': 200, '5-1_alpacafarm': 200}
Baseline originally detected: 583 /600
MarkLLM records verified: 5 ratios x 600 responses


## 3. Lock the experiment configuration and input versions

In [3]:
def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


input_paths = [WM_PATH, NOWM_PATH, BASELINE_PATH, ROOT / "src" / "detection.py"]
input_paths.extend(markllm_paths[r] for r in RATIOS)

config = {
    "attack_version": ATTACK_VERSION,
    "split_seeds": SPLIT_SEEDS,
    "ratios": RATIOS,
    "min_support": MIN_SUPPORT,
    "alpha": ALPHA,
    "methods": METHODS,
    "input_sha256": {str(p.relative_to(ROOT)): file_sha256(p) for p in input_paths},
}
config_path = OUT_DIR / "config.json"
if config_path.exists():
    old_config = json.loads(config_path.read_text(encoding="utf-8"))
    if old_config != config:
        raise RuntimeError(
            "Saved results use a different configuration or input version. "
            "Do not mix experiments; change OUT_DIR to a new folder."
        )
    print("Existing experiment configuration matches: OK")
else:
    config_path.write_text(json.dumps(config, indent=2), encoding="utf-8")
    print("Configuration saved:", config_path)

Configuration saved: /Users/paulmicheli/Desktop/Unine/MA1/Trustworthy/Trustworhty_AI_Watermarking/results/attacks/stealing_guided_v1_improvements/ratio_sweep_support10/config.json


## 4. Train/test split and statistical dictionary (identical rules to V1)

In [4]:
def make_split(seed: int):
    rng = random.Random(seed)
    train, test = [], []

    for j, (_, original_indices) in enumerate(sorted(groups.items())):
        indices = original_indices.copy()
        rng.shuffle(indices)
        n_train = 134 if j == 0 else 133
        train.extend(indices[:n_train])
        test.extend(indices[n_train:])

    train, test = sorted(train), sorted(test)
    assert len(train) == 400 and len(test) == 200
    assert not (set(train) & set(test))
    assert set(train) | set(test) == set(range(600))
    return train, test


def tokenize_words(text: str):
    return WORD_PATTERN.findall(text.lower())


@lru_cache(maxsize=None)
def get_synonyms(word: str):
    """Single-word WordNet lemmas, matching notebook 09's candidate rule."""
    synonyms = set()
    for synset in wordnet.synsets(word):
        for lemma in synset.lemmas():
            candidate = lemma.name().lower()
            if candidate != word and re.fullmatch(r"[a-z]+", candidate):
                synonyms.add(candidate)
    return tuple(sorted(synonyms))


def learn_dictionary(training_indices):
    wm_counts, nowm_counts = Counter(), Counter()
    for idx in training_indices:
        wm_counts.update(tokenize_words(wm[idx]["completion"]))
        nowm_counts.update(tokenize_words(no_wm[idx]["completion"]))

    vocabulary = set(wm_counts) | set(nowm_counts)
    v = len(vocabulary)
    wm_total = sum(wm_counts.values())
    nowm_total = sum(nowm_counts.values())
    scores = {
        word: math.log(
            ((wm_counts[word] + ALPHA) / (wm_total + ALPHA * v))
            / ((nowm_counts[word] + ALPHA) / (nowm_total + ALPHA * v))
        )
        for word in vocabulary
    }

    dictionary = {}
    for word in sorted(vocabulary):
        if wm_counts[word] + nowm_counts[word] < MIN_SUPPORT:
            continue
        if scores[word] <= 0:
            continue
        candidates = [
            candidate for candidate in get_synonyms(word)
            if wm_counts[candidate] + nowm_counts[candidate] >= MIN_SUPPORT
            and candidate in scores
            and scores[candidate] < scores[word]
        ]
        if candidates:
            best = min(candidates, key=lambda s: scores[s])
            dictionary[word] = {
                "replacement": best,
                "gain": scores[word] - scores[best],
            }
    return dictionary


train_42, test_42 = make_split(42)
dict_42 = learn_dictionary(train_42)
print("Seed 42 | train:", len(train_42), "| test:", len(test_42))
print("Seed 42 | dictionary entries:", len(dict_42))
print("Seed 42 | initially detected in test:", sum(
    bool(baseline_by_idx[idx]["detected"]) for idx in test_42
))
assert sum(bool(baseline_by_idx[idx]["detected"]) for idx in test_42) == 195, (
    "Split 42 does not reproduce notebook 09; check group labels / dataset alignment."
)

Seed 42 | train: 400 | test: 200
Seed 42 | dictionary entries: 554
Seed 42 | initially detected in test: 195


## 5. Attack implementations and exact per-response budget matching

In [5]:
def apply_substitutions(text: str, selected):
    """Apply character-span replacements right-to-left, preserving basic casing."""
    attacked = text
    for item in sorted(selected, key=lambda x: x["start"], reverse=True):
        original = attacked[item["start"]:item["end"]]
        replacement = item["replacement"]
        if original.isupper():
            replacement = replacement.upper()
        elif original.istitle():
            replacement = replacement.capitalize()
        attacked = attacked[:item["start"]] + replacement + attacked[item["end"]:]
    return attacked


def guided_attack(text: str, dictionary: dict, ratio: float):
    matches = list(WORD_PATTERN.finditer(text.lower()))
    candidates = []
    for match in matches:
        word = match.group()
        if word in dictionary:
            candidates.append({
                "start": match.start(), "end": match.end(),
                "original": word,
                "replacement": dictionary[word]["replacement"],
                "gain": dictionary[word]["gain"],
            })
    candidates.sort(key=lambda x: x["gain"], reverse=True)
    target = round(len(matches) * ratio)
    selected = candidates[:target]
    return {
        "attacked": apply_substitutions(text, selected),
        "n_words": len(matches),
        "n_targeted": target,
        "n_replaced": len(selected),
        "substitutions": selected,
    }


def random_synonyms_same_positions(text: str, guided_result: dict, seed: int):
    rng = random.Random(seed)
    selected = []
    for item in guided_result["substitutions"]:
        synonyms = get_synonyms(item["original"])
        assert synonyms, f"WordNet synonyms missing for {item['original']}"
        selected.append({
            "start": item["start"], "end": item["end"],
            "original": item["original"],
            "replacement": rng.choice(synonyms),
        })
    return {
        "attacked": apply_substitutions(text, selected),
        "n_words": guided_result["n_words"],
        "n_targeted": guided_result["n_targeted"],
        "n_replaced": len(selected),
        "substitutions": selected,
    }


def fully_random_attack(text: str, n_replacements: int, ratio: float, seed: int):
    rng = random.Random(seed)
    matches = list(WORD_PATTERN.finditer(text.lower()))
    candidates = []
    for match in matches:
        word = match.group()
        synonyms = get_synonyms(word)
        if synonyms:
            candidates.append({
                "start": match.start(), "end": match.end(),
                "original": word, "synonyms": synonyms,
            })

    if n_replacements > len(candidates):
        raise RuntimeError(
            f"Only {len(candidates)} random candidate positions for {n_replacements} edits."
        )
    sampled = rng.sample(candidates, n_replacements)
    selected = [{
        "start": item["start"], "end": item["end"],
        "original": item["original"],
        "replacement": rng.choice(item["synonyms"]),
    } for item in sampled]

    return {
        "attacked": apply_substitutions(text, selected),
        "n_words": len(matches),
        "n_targeted": round(len(matches) * ratio),
        "n_replaced": len(selected),
        "substitutions": selected,
    }

## 6. Sanity checks before full evaluation

In [6]:
# (a) Check original V1 output is reproducible from the saved dictionary rules.
original_guided_file = (
    ROOT / "results" / "attacks" / "stealing_guided" / "stealing_guided_r0.1.jsonl"
)
if original_guided_file.is_file():
    old_guided = {int(r["idx"]): r for r in load_jsonl(original_guided_file)}
    assert set(old_guided) == set(test_42)
    mismatches = [
        idx for idx in test_42
        if guided_attack(wm[idx]["completion"], dict_42, 0.1)["attacked"]
        != old_guided[idx]["attacked"]
    ]
    assert not mismatches, (
        f"New guided code differs from notebook 09 on {len(mismatches)} texts. "
        f"First differences: {mismatches[:5]}"
    )
    print("V1 seed 42, ratio 0.1: all 200 attacked texts match notebook 09.")
else:
    print("Optional V1 output unavailable; no byte-for-byte reproduction check.")

# (b) Check all methods are budget-matched on two TRAIN responses only.
for idx in train_42[:2]:
    original = wm[idx]["completion"]
    for ratio in RATIOS:
        g = guided_attack(original, dict_42, ratio)
        r = random_synonyms_same_positions(original, g, seed=5000 + idx)
        f = fully_random_attack(original, g["n_replaced"], ratio, seed=6000 + idx)
        assert g["n_replaced"] == r["n_replaced"] == f["n_replaced"]
        assert g["n_replaced"] <= g["n_targeted"]
        assert g["n_words"] == r["n_words"] == f["n_words"]
        assert (g["attacked"] == original) == (g["n_replaced"] == 0)
print("All attack budget checks passed (two training texts, five ratios).")

V1 seed 42, ratio 0.1: all 200 attacked texts match notebook 09.
All attack budget checks passed (two training texts, five ratios).


## 7. Evaluation functions and per-split checkpointing

In [7]:
tokenizer, detector = load_detector(ROOT)


def attack_seed(split_seed: int, ratio_index: int, idx: int, method_offset: int):
    # Independent deterministic RNG streams across splits, ratios, texts, and methods.
    return split_seed * 10_000_000 + ratio_index * 1_000_000 + idx * 10 + method_offset


def run_one_split(split_seed: int):
    train_ids, test_ids = make_split(split_seed)
    dictionary = learn_dictionary(train_ids)
    records = []

    for ratio_index, ratio in enumerate(RATIOS):
        markllm_lookup = markllm_by_ratio[ratio]

        for idx in tqdm(test_ids, desc=f"Split {split_seed} | r={ratio:.1f}", leave=False):
            original = wm[idx]["completion"]
            baseline_detected = bool(baseline_by_idx[idx]["detected"])
            n_words = len(tokenize_words(original))

            # Already evaluated in notebook 05. Do NOT recompute its detection.
            markllm = markllm_lookup[idx]
            records.append({
                "split_seed": split_seed, "ratio": ratio, "idx": idx,
                "dataset": dataset_id(wm[idx]), "method": "markllm",
                "original_detected": baseline_detected,
                "detected": bool(markllm["detected"]),
                "p_value": float(markllm["p_value"]),
                "green_fraction": markllm.get("green_fraction"),
                "n_words": n_words,
                "n_targeted": round(n_words * ratio),
                "n_replaced": None,  # MarkLLM: cannot claim its value counts real edits.
                "markllm_reported_n_replaced": int(markllm["n_replaced"]),
                "dictionary_size": len(dictionary),
                "attacked": None,     # Existing notebook 05 file stores this text.
            })

            guided = guided_attack(original, dictionary, ratio)
            random_syn = random_synonyms_same_positions(
                original, guided,
                seed=attack_seed(split_seed, ratio_index, idx, method_offset=1),
            )
            fully_random = fully_random_attack(
                original, guided["n_replaced"], ratio,
                seed=attack_seed(split_seed, ratio_index, idx, method_offset=2),
            )

            assert guided["n_replaced"] == random_syn["n_replaced"] == fully_random["n_replaced"]
            assert guided["n_words"] == random_syn["n_words"] == fully_random["n_words"]

            for method, result in [
                ("guided", guided),
                ("random_synonyms_guided_positions", random_syn),
                ("fully_random", fully_random),
            ]:
                detection = detect(result["attacked"], tokenizer, detector)
                records.append({
                    "split_seed": split_seed, "ratio": ratio, "idx": idx,
                    "dataset": dataset_id(wm[idx]), "method": method,
                    "original_detected": baseline_detected,
                    "detected": bool(detection["detected"]),
                    "p_value": float(detection["p_value"]),
                    "green_fraction": detection.get("green_fraction"),
                    "n_scored": detection.get("n_scored"),
                    "n_tokens": detection.get("n_tokens"),
                    "n_words": result["n_words"],
                    "n_targeted": result["n_targeted"],
                    "n_replaced": result["n_replaced"],
                    "markllm_reported_n_replaced": None,
                    "dictionary_size": len(dictionary),
                    "attacked": result["attacked"],
                })

    assert len(records) == len(RATIOS) * len(test_ids) * len(METHODS)
    return records


def verify_split_records(records, split_seed: int):
    _, expected_test = make_split(split_seed)
    assert len(records) == len(RATIOS) * len(expected_test) * len(METHODS)
    keys = [(r["ratio"], r["idx"], r["method"]) for r in records]
    assert len(keys) == len(set(keys)), "Duplicate split/ratio/text/method result!"
    assert set(r["idx"] for r in records) == set(expected_test)
    assert set(r["ratio"] for r in records) == set(RATIOS)
    assert set(r["method"] for r in records) == set(METHODS)
    assert all(r["split_seed"] == split_seed for r in records)
    for ratio in RATIOS:
        for idx in expected_test:
            by_method = {
                r["method"]: r for r in records
                if r["ratio"] == ratio and r["idx"] == idx
            }
            assert set(by_method) == set(METHODS)
            counts = [by_method[m]["n_replaced"] for m in METHODS if m != "markllm"]
            assert counts[0] == counts[1] == counts[2]
            assert all(
                by_method[m]["original_detected"] == bool(baseline_by_idx[idx]["detected"])
                for m in METHODS
            )
    return True


def save_split_atomic(records, path: Path):
    if path.exists():
        raise FileExistsError(f"Refusing to overwrite completed experiment: {path}")
    temp = path.with_suffix(".tmp.jsonl")
    if temp.exists():
        temp.unlink()  # Stale temporary file from an interrupted save.
    save_jsonl(records, temp)
    temp.rename(path)

/Users/paulmicheli/Desktop/Unine/MA1/Trustworthy/Trustworhty_AI_Watermarking/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 8. Execute all ten splits (checkpoint after each split)

In [8]:
all_records = []
t0 = time.perf_counter()
for split_seed in SPLIT_SEEDS:
    split_path = SPLITS_DIR / f"split_{split_seed}.jsonl"
    if split_path.is_file():
        records = load_jsonl(split_path)
        verify_split_records(records, split_seed)
        print(f"Split {split_seed}: loaded {len(records)} saved results")
    else:
        records = run_one_split(split_seed)
        verify_split_records(records, split_seed)
        save_split_atomic(records, split_path)
        print(f"Split {split_seed}: saved {len(records)} results")
    all_records.extend(records)

assert len(all_records) == len(SPLIT_SEEDS) * len(RATIOS) * 200 * len(METHODS)
print(f"\nComplete: {len(all_records):,} result rows")
print(f"Elapsed this run: {time.perf_counter() - t0:.1f} s")
print("Results directory:", SPLITS_DIR)

Saved 4000 records to /Users/paulmicheli/Desktop/Unine/MA1/Trustworthy/Trustworhty_AI_Watermarking/results/attacks/stealing_guided_v1_improvements/ratio_sweep_support10/splits/split_42.tmp.jsonl
Split 42: saved 4000 results


Saved 4000 records to /Users/paulmicheli/Desktop/Unine/MA1/Trustworthy/Trustworhty_AI_Watermarking/results/attacks/stealing_guided_v1_improvements/ratio_sweep_support10/splits/split_100.tmp.jsonl
Split 100: saved 4000 results


Saved 4000 records to /Users/paulmicheli/Desktop/Unine/MA1/Trustworthy/Trustworhty_AI_Watermarking/results/attacks/stealing_guided_v1_improvements/ratio_sweep_support10/splits/split_101.tmp.jsonl
Split 101: saved 4000 results


Saved 4000 records to /Users/paulmicheli/Desktop/Unine/MA1/Trustworthy/Trustworhty_AI_Watermarking/results/attacks/stealing_guided_v1_improvements/ratio_sweep_support10/splits/split_102.tmp.jsonl
Split 102: saved 4000 results


Saved 4000 records to /Users/paulmicheli/Desktop/Unine/MA1/Trustworthy/Trustworhty_AI_Watermarking/results/attacks/stealing_guided_v1_improvements/ratio_sweep_support10/splits/split_103.tmp.jsonl
Split 103: saved 4000 results


Saved 4000 records to /Users/paulmicheli/Desktop/Unine/MA1/Trustworthy/Trustworhty_AI_Watermarking/results/attacks/stealing_guided_v1_improvements/ratio_sweep_support10/splits/split_104.tmp.jsonl
Split 104: saved 4000 results


Saved 4000 records to /Users/paulmicheli/Desktop/Unine/MA1/Trustworthy/Trustworhty_AI_Watermarking/results/attacks/stealing_guided_v1_improvements/ratio_sweep_support10/splits/split_105.tmp.jsonl
Split 105: saved 4000 results


Saved 4000 records to /Users/paulmicheli/Desktop/Unine/MA1/Trustworthy/Trustworhty_AI_Watermarking/results/attacks/stealing_guided_v1_improvements/ratio_sweep_support10/splits/split_106.tmp.jsonl
Split 106: saved 4000 results


Saved 4000 records to /Users/paulmicheli/Desktop/Unine/MA1/Trustworthy/Trustworhty_AI_Watermarking/results/attacks/stealing_guided_v1_improvements/ratio_sweep_support10/splits/split_107.tmp.jsonl
Split 107: saved 4000 results


Saved 4000 records to /Users/paulmicheli/Desktop/Unine/MA1/Trustworthy/Trustworhty_AI_Watermarking/results/attacks/stealing_guided_v1_improvements/ratio_sweep_support10/splits/split_108.tmp.jsonl
Split 108: saved 4000 results

Complete: 40,000 result rows
Elapsed this run: 26.9 s
Results directory: /Users/paulmicheli/Desktop/Unine/MA1/Trustworthy/Trustworhty_AI_Watermarking/results/attacks/stealing_guided_v1_improvements/ratio_sweep_support10/splits


## 9. Summarize ASR and achieved budgets (per split × ratio × method)

In [9]:
summary_rows = []

for seed in SPLIT_SEEDS:
    _, test_ids = make_split(seed)
    eligible = {idx for idx in test_ids if bool(baseline_by_idx[idx]["detected"])}
    n_eligible = len(eligible)
    assert n_eligible > 0

    for ratio in RATIOS:
        for method in METHODS:
            subset = [
                r for r in all_records
                if r["split_seed"] == seed and r["ratio"] == ratio and r["method"] == method
            ]
            assert len(subset) == 200
            success = sum(
                not r["detected"] for r in subset if r["idx"] in eligible
            )
            n_detected_after = sum(bool(r["detected"]) for r in subset)
            n_words = sum(r["n_words"] for r in subset)

            if method == "markllm":
                actual_replaced = None
                actual_rate = None
                reported_replaced = sum(r["markllm_reported_n_replaced"] for r in subset)
            else:
                actual_replaced = sum(r["n_replaced"] for r in subset)
                actual_rate = actual_replaced / n_words if n_words else 0.0
                reported_replaced = None

            summary_rows.append({
                "split_seed": seed,
                "ratio": ratio,
                "method": method,
                "test_n": len(test_ids),
                "eligible_n": n_eligible,
                "successes": success,
                "asr": success / n_eligible,
                "detected_after_all": n_detected_after,
                "total_words": n_words,
                "actual_replacements": actual_replaced,
                "actual_replacement_ratio": actual_rate,
                "markllm_reported_replacements": reported_replaced,
                "dictionary_size": subset[0]["dictionary_size"],
            })

summary_df = pd.DataFrame(summary_rows)
assert len(summary_df) == 10 * 5 * 4

summary_jsonl = OUT_DIR / "multisplit_ratio_summary.jsonl"
summary_csv = OUT_DIR / "multisplit_ratio_summary.csv"
if not summary_jsonl.exists():
    save_jsonl(summary_rows, summary_jsonl)
else:
    assert load_jsonl(summary_jsonl) == summary_rows, (
        "Summary exists but differs from current results. Investigate before overwriting."
    )
if not summary_csv.exists():
    summary_df.to_csv(summary_csv, index=False)

print("Saved:", summary_jsonl.name, "+", summary_csv.name)
print("Summary rows:", len(summary_df))

Saved 200 records to /Users/paulmicheli/Desktop/Unine/MA1/Trustworthy/Trustworhty_AI_Watermarking/results/attacks/stealing_guided_v1_improvements/ratio_sweep_support10/multisplit_ratio_summary.jsonl
Saved: multisplit_ratio_summary.jsonl + multisplit_ratio_summary.csv
Summary rows: 200


## 10. ASR mean ± std across the ten overlapping splits

In [10]:
stats_df = (
    summary_df.groupby(["ratio", "method"], as_index=False)
    .agg(
        mean_asr=("asr", "mean"),
        std_asr=("asr", "std"),
        min_asr=("asr", "min"),
        max_asr=("asr", "max"),
        mean_actual_ratio=("actual_replacement_ratio", "mean"),
        mean_dictionary_size=("dictionary_size", "mean"),
    )
)

view = stats_df.copy()
for column in ["mean_asr", "std_asr", "min_asr", "max_asr", "mean_actual_ratio"]:
    view[column] = view[column].map(lambda x: f"{x * 100:.2f}%" if pd.notna(x) else "N/A")
view["mean_dictionary_size"] = view["mean_dictionary_size"].round(0).astype(int)

print("ASR statistics (10 splits). MarkLLM effective edit ratio is unknown.")
display(view.sort_values(["ratio", "method"]).reset_index(drop=True))

print("\nASR by nominal ratio (%), averaged across splits:")
asr_pivot = stats_df.pivot(index="ratio", columns="method", values="mean_asr") * 100
display(asr_pivot.round(2))

print("\nActual replacement rates of our three methods (%, averaged across splits):")
budget_pivot = stats_df.pivot(index="ratio", columns="method", values="mean_actual_ratio") * 100
display(budget_pivot.round(2))

ASR statistics (10 splits). MarkLLM effective edit ratio is unknown.


,ratio,method,mean_asr,std_asr,min_asr,max_asr,mean_actual_ratio,mean_dictionary_size
0,0.1,fully_random,3.77%,2.11%,0.00%,7.85%,9.90%,549
1,0.1,guided,6.45%,1.79%,4.06%,8.90%,9.90%,549
2,0.1,markllm,3.92%,1.71%,1.52%,6.09%,N/A,549
3,0.1,random_synonyms_guided_positions,7.06%,2.10%,4.06%,10.66%,9.90%,549
4,0.2,fully_random,9.64%,2.55%,6.74%,14.43%,17.24%,549
5,0.2,guided,14.04%,3.62%,10.36%,21.47%,17.24%,549
6,0.2,markllm,8.61%,1.63%,6.22%,11.68%,N/A,549
7,0.2,random_synonyms_guided_positions,16.04%,2.46%,12.31%,20.62%,17.24%,549
8,0.3,fully_random,12.32%,3.08%,8.21%,17.77%,18.96%,549
9,0.3,guided,17.40%,4.07%,11.92%,24.61%,18.96%,549



ASR by nominal ratio (%), averaged across splits:


method,fully_random,guided,markllm,random_synonyms_guided_positions
ratio,,,,
0.1,3.77,6.45,3.92,7.06
0.2,9.64,14.04,8.61,16.04
0.3,12.32,17.40,18.61,19.14
0.4,13.62,17.86,32.89,19.19
0.5,13.57,17.91,56.24,20.84



Actual replacement rates of our three methods (%, averaged across splits):


method,fully_random,guided,markllm,random_synonyms_guided_positions
ratio,,,,
0.1,9.90,9.90,NaN,9.90
0.2,17.24,17.24,NaN,17.24
0.3,18.96,18.96,NaN,18.96
0.4,19.06,19.06,NaN,19.06
0.5,19.06,19.06,NaN,19.06


## 11. Paired descriptive differences and success counts

In [11]:
paired = summary_df.pivot(index=["split_seed", "ratio"], columns="method", values="asr")
paired["guided_minus_markllm"] = paired["guided"] - paired["markllm"]
paired["guided_minus_fully_random"] = paired["guided"] - paired["fully_random"]
paired["guided_minus_random_synonyms"] = (
    paired["guided"] - paired["random_synonyms_guided_positions"]
)
paired["random_synonyms_minus_fully_random"] = (
    paired["random_synonyms_guided_positions"] - paired["fully_random"]
)

rows = []
for ratio in RATIOS:
    subset = paired.xs(ratio, level="ratio")
    for contrast in [
        "guided_minus_markllm",
        "guided_minus_fully_random",
        "guided_minus_random_synonyms",
        "random_synonyms_minus_fully_random",
    ]:
        values = subset[contrast]
        rows.append({
            "ratio": ratio,
            "contrast": contrast,
            "mean_difference_pp": 100 * values.mean(),
            "std_difference_pp": 100 * values.std(ddof=1),
            "positive_splits": int((values > 0).sum()),
            "ties": int((values == 0).sum()),
        })

contrasts_df = pd.DataFrame(rows)
display(contrasts_df.round(2))

# Useful for later paired tests; not a hypothesis test here.
print("Note: overlapping splits -> these descriptive statistics are NOT independent-trial p-values.")

,ratio,contrast,mean_difference_pp,std_difference_pp,positive_splits,ties
0,0.1,guided_minus_markllm,2.53,0.75,10,0
1,0.1,guided_minus_fully_random,2.68,1.18,10,0
2,0.1,guided_minus_random_synonyms,-0.61,1.21,2,2
3,0.1,random_synonyms_minus_fully_random,3.29,1.73,9,0
4,0.2,guided_minus_markllm,5.43,3.54,9,0
5,0.2,guided_minus_fully_random,4.40,2.48,10,0
6,0.2,guided_minus_random_synonyms,-2.00,2.62,3,0
7,0.2,random_synonyms_minus_fully_random,6.39,2.13,10,0
8,0.3,guided_minus_markllm,-1.22,4.26,4,1
9,0.3,guided_minus_fully_random,5.07,4.12,8,1


Note: overlapping splits -> these descriptive statistics are NOT independent-trial p-values.


## 12. Curves, dataset breakdown, and export

In [14]:

# Results by ratio — no Matplotlib required
print("=== ASR BY RATIO ===")

summary_table = stats_df.pivot(
    index="ratio",
    columns="method",
    values="mean_asr"
)

print((summary_table * 100).round(2).to_string())

# Per-dataset evaluation
df = pd.DataFrame(all_records)

# Only originally detected responses
df = df[df["original_detected"] == True].copy()
df["success"] = ~df["detected"].astype(bool)

dataset_df = (
    df.groupby(
        ["split_seed", "ratio", "method", "dataset"],
        as_index=False
    )
    .agg(
        eligible_n=("success", "size"),
        successes=("success", "sum")
    )
)

dataset_df["asr"] = (
    dataset_df["successes"] / dataset_df["eligible_n"]
)

dataset_summary = (
    dataset_df.groupby(
        ["dataset", "ratio", "method"],
        as_index=False
    )
    .agg(
        mean_asr=("asr", "mean"),
        std_asr=("asr", "std")
    )
)

dataset_csv = OUT_DIR / "dataset_breakdown.csv"
dataset_summary.to_csv(dataset_csv, index=False)

print("\nPer-dataset summary saved:", dataset_csv)
print("\nAnalysis completed.")


=== ASR BY RATIO ===
method  fully_random  guided  markllm  random_synonyms_guided_positions
ratio                                                                  
0.1             3.77    6.45     3.92                              7.06
0.2             9.64   14.04     8.61                             16.04
0.3            12.32   17.40    18.61                             19.14
0.4            13.62   17.86    32.89                             19.19
0.5            13.57   17.91    56.24                             20.84

Per-dataset summary saved: /Users/paulmicheli/Desktop/Unine/MA1/Trustworthy/Trustworhty_AI_Watermarking/results/attacks/stealing_guided_v1_improvements/ratio_sweep_support10/dataset_breakdown.csv

Analysis completed.


## Interpretation checklist (after running)

1. **Check the `actual_replacement_ratio` table first.** When the dictionary cannot supply enough substitutions, `ratio=0.5` does **not** mean that 50% of words were changed.
2. **Use notebook 05 / MarkLLM as the main reproduced reference**, but report its replacement budget as *nominal / reported*, not necessarily exact.
3. **Same positions, random synonyms** isolates the effect of the statistical *choice of synonym* at the guided positions.
4. **Fully random** isolates the effect of the guided *position selection* only approximately, because it also draws from a larger set of WordNet-eligible words; this is a methodological limitation.
5. **No semantic quality claim.** WordNet can radically alter meaning and grammar.
6. **No p < 0.01 claim from the ten overlapping splits.** They provide a robustness analysis; a separate paired significance analysis and confidence intervals remain future work.
7. **No parameter selection on these held-out tests.** If trying `MIN_SUPPORT = 20/50`, new scoring rules or model tuning, use a validation subset within training data and report the remaining results as exploratory until a new untouched test is available.

**Saved outputs:** `config.json`, one checkpoint JSONL per split, the summary JSONL/CSV, an ASR plot and a per-dataset CSV under `results/attacks/stealing_guided_v1_improvements/ratio_sweep_support10/`.